## Add immune-cell annotations to GSE186328

In [ ]:
from pathlib import Path

import pandas as pd
import scanpy as sc
import anndata as ad
from scipy import sparse
from scipy.stats import median_abs_deviation
import numpy as np
import celltypist
from celltypist import models

In [ ]:
base_dir = Path("path/to/your/directory")  # Change to your working directory
csv_path = base_dir / "GSE186328_raw_counts.csv"
output_h5ad = base_dir / "GSE186328_selected_raw.h5ad"

In [ ]:
# =========================================================
# Read in adata
# =========================================================
print(f"\n[INFO] Reading back saved file: {output_h5ad}")
adata = sc.read_h5ad(output_h5ad)
print(adata)
print("\n[INFO] Patient cell counts from saved file:")
print(adata.obs["patient_id"].value_counts())

In [6]:
model = models.Model.load(model = 'Immune_All_Low.pkl')

In [8]:
# Save the counts layer to a new layer called "raw_counts"
adata.layers["raw_counts"] = adata.X.copy()

In [9]:
# Normalize to 10,000 counts per cell
sc.pp.normalize_total(adata, target_sum=1e4)

# Natural log1p transform
sc.pp.log1p(adata)

In [ ]:
predictions = celltypist.annotate(adata, model = 'Immune_All_Low.pkl', majority_voting = True)
adata = predictions.to_adata(prefix="typist_immune_", insert_conf_by="majority_voting")
print(adata.obs["typist_immune_majority_voting"].head())
print(adata.obs["typist_immune_conf_score"].head())

In [14]:
score_col = "typist_immune_conf_score"

# Print head of this column
print(adata.obs[score_col].head())
# Print number of cells in adata
print(adata.n_obs)

# Filter adata for score_col > 0.5
adata = adata[adata.obs[score_col] > 0.5]
print(f"After filtering for {score_col} > 0.5, number of cells in adata: {adata.n_obs}")


lv2_AAACCCAGTAACGATA    0.998936
lv2_AAACCCATCCCATAGA    0.324689
lv2_AAACCCATCTCCTACG    0.905800
lv2_AAACGCTAGGTTACAA    0.760311
lv2_AAACGCTTCTTGTGCC    0.000832
Name: typist_immune_conf_score, dtype: float64
43031
After filtering for typist_immune_conf_score > 0.5, number of cells in adata: 32024


In [15]:
# See the counts of typist_immune_majority_voting
print(adata.obs["typist_immune_majority_voting"].value_counts())

typist_immune_majority_voting
Classical monocytes              7988
Tem/Trm cytotoxic T cells        5470
MAIT cells                       5421
CD16+ NK cells                   4613
CD16- NK cells                   3987
Tem/Temra cytotoxic T cells      1204
CRTAM+ gamma-delta T cells       1061
Tem/Effector helper T cells       582
Memory B cells                    512
Naive B cells                     369
Non-classical monocytes           140
Regulatory T cells                128
Trm cytotoxic T cells             113
Type 17 helper T cells             97
Tcm/Naive helper T cells           91
Macrophages                        68
Endothelial cells                  50
Plasma cells                       33
Neutrophils                        29
DC1                                24
Erythrophagocytic macrophages      24
B cells                            20
Name: count, dtype: int64


In [16]:
cell_types_to_adata = [
    "CD16- NK cells",
    "CD16+ NK cells",
    "Tem/Trm cytotoxic T cells",
    "MAIT cells",
    "Memory B cells",
    "Tem/Temra cytotoxic T cells",
    "Tem/Effector helper T cells",
    "Naive B cells",
    "Classical monocytes",
    "NK cells",
    "Non-classical monocytes",
    "Endothelial cells",
    "Tcm/Naive helper T cells",
    "Kupffer cells",
    "CRTAM+ gamma-delta T cells",
    "pDC",
]

immune_dir = base_dir / "immune"
immune_dir.mkdir(exist_ok=True, parents=True)

existing_labels = set(adata.obs["typist_immune_majority_voting"].astype(str).unique())
celltype_adatas = {}

for cell_type in cell_types_to_adata:
    if cell_type not in existing_labels:
        continue

    subset = adata[adata.obs["typist_immune_majority_voting"] == cell_type].copy()
    celltype_adatas[cell_type] = subset

    safe_name = (
        cell_type.replace("/", "_")
        .replace(" ", "_")
    )
    subset.write_h5ad(immune_dir / f"{safe_name}.h5ad")

In [ ]:
# =========================================================
# Save AnnData object to disk
# =========================================================
output_h5ad = base_dir / "GSE186328_selected_annotated.h5ad"
print(f"\n[INFO] Saving to: {output_h5ad}")
output_h5ad.parent.mkdir(parents=True, exist_ok=True)
adata.write_h5ad(output_h5ad, compression="gzip")
print("[INFO] Done.")